# Welcome to Graph Rewrite

**Graph Rewrite** is a Python library for performing graph transformations using a declarative approach. It provides a framework for transforming graphs by finding and rewriting subgraphs that match specific patterns. This approach combines structure-based matching with attribute filtering and advanced filtering options, making it suitable for rewriting subgraphs in ASTs, execution graphs, term graphs, and other graph types to improve efficiency.

The library includes extended features that go beyond standard graph rewriting, enabling the efficient identification of complex patterns within input graphs.

## Installation

```bash
#Install package given a conda environment with python>3.9
git clone https://github.com/DeanLight/graph_rewrite
cd graph_rewrite
pip install -e .

```

## Docker

```bash
cd graph_rewrite
# build container
docker-compose build

# spin up the container
docker-compose up

# get a bash terminal on a spun up container
docker-compose exec main bash

# spin up and get a bash terminal (closing it will close the container)
docker-compose run main bash

```

## Getting started

### Requirements

To use **Graph Rewrite**, you’ll need to have `networkx` installed, as the input graph should be in a `networkx.DiGraph` format.
Make sure to include the following imports at the start of your script or notebook:

In [92]:
import networkx as nx
from graph_rewrite.transform import rewrite

### The Rewrite Function

The library interface is built around a single main function, `rewrite`. This function lets you define patterns, find matches, and transform subgraphs in an input graph according to specific rules you set.

To use `rewrite`, you provide three pattern strings: *LHS*, *P*, and *RHS*. These strings describe the patterns you want to match in the input graph and the transformations to apply. You can also add optional parameters to customize how matches are found and transformed.

This interface is flexible enough to support a wide range of graph transformation needs.

### Using LHS, P, and RHS Patterns in Graph Rewrite

The `rewrite` function in the Graph Rewrite library uses three key components to define and apply transformations to graphs: **LHS**, **P**, and **RHS** patterns. These patterns work together to match and rewrite subgraphs within the input graph.

- **LHS (Left-Hand Side)**: Defines the subgraph structure we’re searching for in the input graph. This represents the "before" state and specifies the exact pattern we’re looking to match prior to rewriting.
- **P (Preserve)**: Defines which parts of the matched subgraph should remain unchanged after the rewrite. This can include specific nodes, edges, and their attributes. If an empty P string is passed to the *rewrite* function, no objects from the matched subgraph will be removed.
- **RHS (Right-Hand Side)**: Defines the new structure to create based on the matched subgraph, allowing us to modify or add elements within the graph.

## Usecases 

Here are some simple use cases that demonstrate how LHS, P, and RHS work together to define different transformations:

#### 1. Basic Attribute Replacement

We start with an input directed graph (`input_graph`) containing a single node `'A'` with a label attribute set to `'Node_A'`.

Using the following strings:
* **LHS**: `lhs = 'x[label="Node_A"]'`, specifying a search for any node with a label attribute equal to `'Node_A'`. Here, `'x'` is a symbolic name for the matching node.
* **P**: `p = 'x[label]'`, indicating that we want to preserve the node `'x'` and retain its label attribute. This prevents the node from being removed during the rewrite but allows its label value to be updated.
* **RHS**: `rhs = 'x[label="Node_B"]'`, specifying that we want to change the label of the node `'x'` to `'Node_B'`.

We call `rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)` to apply the transformation. This finds the node `'A'` with `label="Node_A"`, preserves it, and updates the label to `"Node_B"`. 

After the rewrite, `input_graph.nodes(data=True)` should print `[('A', {'label': 'Node_B'})]`, confirming that the label of node `'A'` was successfully updated to `'Node_B'`.

In [93]:
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A', {'label': 'Node_A'})])
lhs = 'x[label="Node_A"]'
p = 'x[label]'
rhs = 'x[label="Node_B"]'
rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)
print(input_graph.nodes(data=True)) # Should print [('A', {'label': 'Node_B'})]

[('A', {'label': 'Node_B'})]


#### 2. Removing an Intermediate Node and Adding a Direct Edge 

In this test, we start with a directed graph (`input_graph`) that contains a path `A -> B -> C`. Using the LHS, P, and RHS strings, we’ll find this pattern, remove the intermediate node `B`, and add a direct edge from `A` to `C`.

We use the following strings:
* **LHS**: The string `lhs = 'x->y->z'` specifies that we’re looking for a path where node `x` connects to node `y`, which in turn connects to node `z`. Here, `x`, `y`, and `z` are symbolic names for the matching nodes in the input graph.
* **P**: The string `p = 'x->z'` specifies that we want to preserve the direct path from `x` to `z`, which will be created during the rewrite. This also implies that `y` is removed from the graph during the rewrite since it’s not part of `P`.
* **RHS**: The string `rhs = 'x->z'` specifies that we want to create a new edge directly from `x` to `z` after removing `y`.

By calling `rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)`, we apply the transformation. The function will find the path `A -> B -> C`, remove the intermediate node `B`, and create a direct edge from `A` to `C`.

After the rewrite, `input_graph.edges()` should print `[('A', 'C')]`, showing that the transformation successfully removed `B` and added the edge `A -> C`.

In [94]:
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A', {'label': 'Node_A'}), ('B', {'label': 'Node_B'}), ('C', {'label': 'Node_C'})])
input_graph.add_edges_from([('A', 'B'), ('B', 'C')])
lhs = 'x->y->z'
p = 'x,z'
rhs = 'x->z'
rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs)
print(list(input_graph.edges()))  # Should print [('A', 'C')]

[('A', 'C')]


#### 3. Finding and Transforming a Parent-Child Pair with Attribute Modification

In this example, we start with a graph where each parent node connects to a child node. The goal is to find all such pairs where the parent node has an attribute `'val'`, increment this `'val'` by 1, and store the result in the connecting edge.

We use the following strings:
* **LHS**: The string `lhs = 'a[val]->b'` specifies that we are looking for an edge from node `'a'` to node `'b'` where `'a'` has an attribute `'val'`.
* **P**: The string `p = 'a->b'` specifies that we want to preserve both nodes `'a'` and `'b'` and the edge between them.
* **RHS**: The string `rhs = 'a-[val={{new_val}}]->b'` specifies that we want to add an edge attribute `'val'` to the edge from `'a'` to `'b'`, which is calculated by incrementing the existing `'val'` attribute of node `'a'` by 1.

We also provide a custom `render_rhs` function to calculate the new value for the edge. This function uses the value of `'val'` in `'a'`, increments it by 1, and stores it in the edge between `'a'` and `'b'`.

After applying `rewrite`, the transformed graph should have the edge from `'a'` to `'b'` updated with the new `'val'` attribute.

In [95]:
input_graph = nx.DiGraph()
input_graph.add_node('A', val=10)  # Example initial value
input_graph.add_node('B')
input_graph.add_edge('A', 'B')
lhs = 'a[val]->b'
p = 'a->b'
rhs = 'a-[val={{new_val}}]->b'
rewrite(
    input_graph=input_graph,
    lhs=lhs,
    p=p,
    rhs=rhs,
    render_rhs={'new_val': lambda match: match['a']['val'] + 1}
)
print(input_graph.edges(data=True))  # Should print [('A', 'B', {'val': 11})]

[('A', 'B', {'val': 11})]


### 4. Using the Collections Feature To Create a List of All Grandchildren

This test demonstrates how to use the collections feature to capture multiple matches for a node in the input graph. Starting with a graph containing nodes `'A'`, `'B'`, `'C'`, `'D'`, and `'E'`, connected as follows: `A -> B`, `A -> C`, `B -> D`, and `B -> E`.

We define the transformation using the following patterns:
* **LHS**: `lhs = 'x->y;y->z'` specifies a pattern where **x** connects to **y**, and **y** has at least one outgoing edge to **z**. Here, **x** and **y** represent unique nodes in each match, while **z** represents a collection of nodes connected to **y**.
* **P**: `p = 'x->y,y->z'` specifies that we want to preserve the edges from **x** to **y** and from **y** to each **z** node in the collection.
* **RHS**: `rhs = 'x[grandchildren={{new_val}}]->y,y->z'` specifies that we want to add an attribute `grandchildren` to **x**. This attribute will be a list containing the labels of all nodes in the **z** collection, captured by `{{new_val}}`.

We apply `rewrite` with a custom function in `render_rhs` that gathers the labels of each **z** node in the collection. This transformation matches the structure `A -> B -> {D, E}` and updates **A** to include a `grandchildren` attribute listing `['Node_D', 'Node_E']`.

In [96]:
input_graph = nx.DiGraph()
input_graph.add_nodes_from([('A', {'label': 'Node_A'}), ('B', {'label': 'Node_B'}), ('C', {'label': 'Node_C'}), ('D', {'label': 'Node_D'}), ('E', {'label': 'Node_E'})])
input_graph.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'D'), ('B', 'E')])

lhs = 'x->y;y->z'
p = 'x->y,y->z'
rhs = 'x[grandchildren={{new_val}}]->y,y->z'

rewrite(input_graph=input_graph, lhs=lhs, p=p, rhs=rhs, render_rhs={'new_val': lambda match: match['z']['label']})

# After the rewrite, 'A' should have the label with the count of z nodes.
print(input_graph.nodes['A'])  # Should print {'label':  'Node_A', 'grandchildren': ['Node_D', 'Node_E']}

{'label': 'Node_A', 'grandchildren': ['Node_D', 'Node_E']}
